# Set up

In [1]:
project_folder = "/content/drive/MyDrive/weather_predictor"

from google.colab import drive
drive.mount('/content/drive')

%cd "{project_folder}"
!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/weather_predictor
Already up to date.


# Cài đặt môi trường

In [105]:
!git pull
!pip install -r requirements.txt

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 270 bytes | 0 bytes/s, done.
From https://github.com/zalexdevon/weather_predictor
   4f5cb9b..04a90db  main       -> origin/main
Updating 4f5cb9b..04a90db
Fast-forward
 requirements.txt | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
Obtaining file:///content/drive/MyDrive/weather_predictor (from -r requirements.txt (line 16))
  Preparing metadata (setup.py) ... done
  Attempting uninstall: zzzz_mylib_23_4
    Found existing installation: zzzz_mylib_23_4 0.28
    Uninstalling zzzz_mylib_23_4-0.28:
      Successfully uninstalled zzzz_mylib_23_4-0.28
  Attempting uninstall: classifier
    Found existing installation: classifier 0.0.0
    Uninstalling classifier-0.0.0:
      Successfully uninstalled classifier-0.0.0
  Running setup.py develop for classifier

# Thư viện

In [2]:
import pandas as pd
import numpy as np
from Mylib import myfuncs
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import plotly.express as px
import re
import os
from plotly.subplots import make_subplots

# r1

In [ ]:
dt1 = myfuncs.load_python_object("artifacts/data_transformation/dt1_2/train_features.pkl")

pre = PolynomialFeatures(degree=3, include_bias=False)

dt1_pre = pre.fit_transform(dt1)

dt1_pre.shape

(47105, 3653)

# dc1

## Đọc dữ liệu


In [ ]:
df = myfuncs.load_python_object("artifacts/data_ingestion/train_data.pkl")

df.head()


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,temp_bin
56032,Luxembourg,Luxembourg,49.6117,6.1300,Europe/Luxembourg,1740822300,2025-03-01 10:45,3.0,37.4,Partly cloudy,...,23.495,2,2,07:18 AM,06:18 PM,07:55 AM,08:30 PM,Waxing Crescent,1,low
34107,Sweden,Stockholm,59.3333,18.0500,Europe/Stockholm,1731056400,2024-11-08 10:00,7.3,45.1,Sunny,...,29.230,2,2,07:26 AM,03:37 PM,02:20 PM,08:48 PM,Waxing Crescent,37,low
54115,Palau,Airai,7.3575,134.5578,Pacific/Palau,1739960100,2025-02-19 19:15,27.3,81.1,Overcast,...,9.065,1,1,06:18 AM,06:13 PM,11:15 PM,10:29 AM,Waning Gibbous,69,medium
9861,South Africa,Pretoria,-25.7500,28.1900,Africa/Johannesburg,1720098900,2024-07-04 15:15,19.1,66.4,Sunny,...,13.800,1,1,06:54 AM,05:29 PM,05:33 AM,03:57 PM,Waning Crescent,5,low
65611,Macedonia,Skopje,42.0000,21.4333,Europe/Skopje,1745053200,2025-04-19 11:00,21.3,70.3,Partly Cloudy,...,13.320,1,1,05:48 AM,07:20 PM,01:04 AM,09:27 AM,Waning Gibbous,71,medium


Kich thuoc


In [ ]:
df.shape


(52542, 42)

## Ý nghĩa các cột

| Cot                       | Y nghia                                                               | Don vi | Phan loai |
| ------------------------- | --------------------------------------------------------------------- | ------ | --------- |
| **country**                     | Đất nước mà dữ liệu được đo                                     | none   | Nominal   |
| **location_name**                       | Tên thành phố                                                          | none   | Nominal   |
| **latitude**             | vĩ độ của thành phố                                                        | none   | ordinal   |
| **longitude**             | kinh độ của thành phố                                                        | none   | ordinal   |
| **timezone**             | Giờ khu vực                                                        | none   | ordinal   |
| **last_updated_epoch**            |  thời điểm cập nhật dữ liệu cuối cùng dưới dạng Unix timestamp.       | none   | ordinal   |
| **last_updated**            |  thời điểm cập nhật dữ liệu cuối cùng (local time)       | none   | ordinal   |
| **temperature_celsius**            |  nhiệt độ dưới dạng C       | none   | ordinal   |
| **temperature_fahrenheit**            |  nhiệt độ dưới dạng F      | none   | ordinal   |
| **condition_text**            |  Tình trạng thời tiết      | none   | ordinal   |
| **wind_mph**         | Tốc độ gió (mile / hour)      | none   | ordinal   |
| wind_kph         | Tốc độ gió (km / hour)      | none   | numeric   |
| wind_degree        | Hướng gió theo độ    | none   | numeric   |
| **wind_direction**         | Hướng gió chi tiết hơn     | none   | numeric   |
| **pressure_mb**         | Áp suất (milibars)    | none   | numeric   |
| pressure_in         | Áp suất (inches)    | none   | numeric   |
| **precip_mm**         | Lượng mưa (mm)    | none   | numeric   |
| **precip_in**         | Lượng mưa (inches)    | none   | numeric   |
| humidity         | Độ ẩm (%)    | none   | numeric   |
| cloud         | Phần trăm mây bao phủ (%)    | none   | numeric   |
| **feels_like_celsius**         | Nhiệt độ cơ thể cảm nhận được thay vì là thực tế    | none   | numeric   |
| **feels_like_fahrenheit**         | Nhiệt độ cơ thể cảm nhận được thay vì là thực tế    | none   | numeric   |
| visibility_km         | Tầm nhìn (km)    | none   | numeric   |
| **visibility_miles**         | Tầm nhìn (mile)    | none   | numeric   |
| uv_index         | Chỉ số tia UV    | none   | numeric   |
| **gust_mph**         | sự tăng tốc đột ngột và mạnh mẽ của gió (mile / h)    | none   | numeric   |
| gust_kph         | sự tăng tốc đột ngột và mạnh mẽ của gió (km / h)    | none   | numeric   |
| air_quality_Carbon_Monoxide         | Nồng độ $CO$    | none   | numeric   |
| air_quality_Ozone         | Nồng độ $O_3$    | none   | numeric   |
| air_quality_Nitrogen_dioxide         | Nồng độ $NO_2$    | none   | numeric   |
| air_quality_Sulphur_dioxide         | Nồng độ $SO_2$    | none   | numeric   |
| air_quality_PM2.5         | Nồng độ $PM2.5$    | none   | numeric   |
| air_quality_PM10         | Nồng độ $PM10$    | none   | numeric   |
| **air_quality_us-epa-index**         | Chỉ số AQI    | none   | numeric   |
| **air_quality_gb-defra-index**         | Chỉ số AQI    | none   | numeric   |
| sunrise         | Thời điểm mặt trời lên   | none   | ordinal   |
| sunset         | Thời điểm mặt trời lặn   | none   | ordinal   |
| moonrise         | Thời điểm trăng lên   | none   | ordinal   |
| moonset         | Thời điểm trăng lặn   | none   | ordinal   |
| moon_phase         | Pha của mặt trăng   | none   | nominal   |
| moon_illumination         | Độ sáng của mặt trăng (%)   | none   | numeric |


## Xóa các cột không cần thiết




### Xóa các cột

In [ ]:
df = df.drop(
    columns=[
        "country",
        "location_name",
        "latitude",
        "longitude",
        "timezone",
        "last_updated_epoch",
        "last_updated",
        "temperature_celsius",
        "temperature_fahrenheit",
        "condition_text",
        "wind_mph",
        "wind_direction",
        "pressure_mb",
        "precip_mm",
        "precip_in",
        "feels_like_celsius",
        "feels_like_fahrenheit",
        "visibility_miles",
        "gust_mph",
        "air_quality_us-epa-index",
        "air_quality_gb-defra-index",
    ]
)

df.columns

Index(['wind_kph', 'wind_degree', 'pressure_in', 'humidity', 'cloud',
       'visibility_km', 'uv_index', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase',
       'moon_illumination', 'temp_bin'],
      dtype='object')

### Tỉ lệ missing các cột

In [ ]:
null_percent = df.isnull().mean() * 100
null_percent = null_percent.sort_values(ascending=False)
null_percent


,0
wind_kph,0.0
air_quality_Sulphur_dioxide,0.0
moon_illumination,0.0
moon_phase,0.0
moonset,0.0
moonrise,0.0
sunset,0.0
sunrise,0.0
air_quality_PM10,0.0
air_quality_PM2.5,0.0


### Xóa các cột có tỉ lệ missing lớn

Ti le missing của các cột đều = 0-> Khong xoa cot nao het !


In [ ]:
df.shape


(52542, 21)

## Đổi tên cột

In [ ]:
rename_dict = {
    "wind_kph": "wind_kph_num",
    "wind_degree": "wind_degree_num",
    "pressure_in": "pressure_in_num",
    "humidity": "humidity_num",
    "cloud": "cloud_num",
    "visibility_km": "visibility_km_num",
    "uv_index": "uv_index_num",
    "gust_kph": "gust_kph_num",
    "air_quality_Carbon_Monoxide": "air_quality_Carbon_Monoxide_num",
    "air_quality_Ozone": "air_quality_Ozone_num",
    "air_quality_Nitrogen_dioxide": "air_quality_Nitrogen_dioxide_num",
    "air_quality_Sulphur_dioxide": "air_quality_Sulphur_dioxide_num",
    "air_quality_PM2.5": "air_quality_PM2_5_num",
    "air_quality_PM10": "air_quality_PM10_num",
    "sunrise": "sunrise_ord",
    "sunset": "sunset_ord",
    "moonrise": "moonrise_ord",
    "moonset": "moonset_ord",
    "moon_phase": "moon_phase_nom",
    "moon_illumination": "moon_illumination_num",
    "temp_bin": "temp_bin_target",

}


df = df.rename(columns=rename_dict)

df.columns


Index(['wind_kph_num', 'wind_degree_num', 'pressure_in_num', 'humidity_num',
       'cloud_num', 'visibility_km_num', 'uv_index_num', 'gust_kph_num',
       'air_quality_Carbon_Monoxide_num', 'air_quality_Ozone_num',
       'air_quality_Nitrogen_dioxide_num', 'air_quality_Sulphur_dioxide_num',
       'air_quality_PM2_5_num', 'air_quality_PM10_num', 'sunrise_ord',
       'sunset_ord', 'moonrise_ord', 'moonset_ord', 'moon_phase_nom',
       'moon_illumination_num', 'temp_bin_target'],
      dtype='object')

## Sắp xếp các cột theo đúng thứ tự

In [ ]:
numeric_cols, numericCat_cols, cat_cols, binary_cols, nominal_cols, ordinal_cols, target_col = myfuncs.get_different_types_cols_from_df_4(df)


df = df[
    numeric_cols
    + numericCat_cols
    + binary_cols
    + nominal_cols
    + ordinal_cols
    + [target_col]
]


df.head()


,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,...,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,sunrise_ord,sunset_ord,moonrise_ord,moonset_ord,temp_bin_target
56032,18.0,57,30.39,81,75,10.0,1.3,24.2,384.80,52.0,...,0.925,21.090,23.495,1,Waxing Crescent,07:18 AM,06:18 PM,07:55 AM,08:30 PM,low
34107,10.1,247,30.55,80,2,10.0,0.3,17.0,362.60,26.0,...,0.925,17.575,29.230,37,Waxing Crescent,07:26 AM,03:37 PM,02:20 PM,08:48 PM,low
54115,24.5,75,29.83,89,100,19.0,0.0,33.6,177.60,31.0,...,0.925,5.920,9.065,69,Waning Gibbous,06:18 AM,06:13 PM,11:15 PM,10:29 AM,medium
9861,22.3,211,30.02,12,0,10.0,6.0,25.7,240.30,76.5,...,3.500,5.700,13.800,5,Waning Crescent,06:54 AM,05:29 PM,05:33 AM,03:57 PM,low
65611,3.6,256,29.94,40,0,10.0,6.0,3.7,264.55,80.0,...,6.290,8.140,13.320,71,Waning Gibbous,05:48 AM,07:20 PM,01:04 AM,09:27 AM,medium


## Kiểm tra kiểu dữ liệu các cột

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 52542 entries, 56032 to 4001
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   wind_kph_num                      52542 non-null  float64 
 1   wind_degree_num                   52542 non-null  int64   
 2   pressure_in_num                   52542 non-null  float64 
 3   humidity_num                      52542 non-null  int64   
 4   cloud_num                         52542 non-null  int64   
 5   visibility_km_num                 52542 non-null  float64 
 6   uv_index_num                      52542 non-null  float64 
 7   gust_kph_num                      52542 non-null  float64 
 8   air_quality_Carbon_Monoxide_num   52542 non-null  float64 
 9   air_quality_Ozone_num             52542 non-null  float64 
 10  air_quality_Nitrogen_dioxide_num  52542 non-null  float64 
 11  air_quality_Sulphur_dioxide_num   52542 non-null  float6

SAI:

- cac cot nominal, ordinal


### Chuyển kdl = kdl mong muốn + NAN


In [ ]:
for col in df.columns.tolist()[15:]:
  print(f"{col} -> {set(map(type, df[col]))}")


moon_phase_nom -> {<class 'str'>}
sunrise_ord -> {<class 'str'>}
sunset_ord -> {<class 'str'>}
moonrise_ord -> {<class 'str'>}
moonset_ord -> {<class 'str'>}
temp_bin_target -> {<class 'str'>}


- Tất cả các cột đều đúng kiểu dữ liệu


## Kiểm tra nội dung các cột `binary`


In [ ]:
for col in binary_cols:
  print(f"{col} -> {df[col].unique().tolist()}")

Không có cột nào hết


## Kiểm tra nội dung các cột `nominal`


In [ ]:
for col in nominal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


moon_phase_nom -> ['Waxing Crescent', 'Waning Gibbous', 'Waning Crescent', 'Last Quarter', 'Waxing Gibbous', 'First Quarter', 'Full Moon', 'New Moon']


## Kiểm tra nội dung các cột `ordinal`


In [ ]:
for col in ordinal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


sunrise_ord -> ['07:18 AM', '07:26 AM', '06:18 AM', '06:54 AM', '05:48 AM', '05:45 AM', '07:13 AM', '06:31 AM', '05:32 AM', '06:14 AM', '05:17 AM', '06:03 AM', '06:52 AM', '06:40 AM', '06:41 AM', '06:50 AM', '05:16 AM', '07:30 AM', '07:56 AM', '06:22 AM', '05:38 AM', '07:17 AM', '06:42 AM', '06:57 AM', '05:33 AM', '06:20 AM', '05:41 AM', '05:23 AM', '06:07 AM', '05:57 AM', '06:00 AM', '04:56 AM', '08:43 AM', '06:13 AM', '05:19 AM', '06:34 AM', '07:22 AM', '04:44 AM', '06:19 AM', '06:09 AM', '05:22 AM', '05:30 AM', '06:05 AM', '05:59 AM', '05:54 AM', '06:27 AM', '05:42 AM', '06:12 AM', '07:45 AM', '06:48 AM', '05:40 AM', '06:32 AM', '06:53 AM', '07:19 AM', '05:56 AM', '05:49 AM', '06:17 AM', '07:03 AM', '06:37 AM', '06:08 AM', '06:16 AM', '07:37 AM', '06:43 AM', '07:16 AM', '06:38 AM', '06:01 AM', '05:18 AM', '05:36 AM', '06:39 AM', '05:39 AM', '05:43 AM', '07:54 AM', '04:06 AM', '07:05 AM', '07:31 AM', '07:59 AM', '07:42 AM', '07:50 AM', '06:45 AM', '05:21 AM', '06:06 AM', '05:51 AM', 

### Biển đổi 4 cột sunrise_ord, sunset_ord, moonrise_ord, moonset_ord

Chuyển các giá trị từ dạng giờ:phút sang dạng khoảng thời gian

VD: 07:03 AM -> 7 - 8 (khoảng thời gian từ 7 - 8h)

In [ ]:
format = r"\d+:\d+\s*(AM|PM)"

df_ordinal_cols = df[ordinal_cols]
df_ordinal_cols.head()

,sunrise_ord,sunset_ord,moonrise_ord,moonset_ord
56032,07:18 AM,06:18 PM,07:55 AM,08:30 PM
34107,07:26 AM,03:37 PM,02:20 PM,08:48 PM
54115,06:18 AM,06:13 PM,11:15 PM,10:29 AM
9861,06:54 AM,05:29 PM,05:33 AM,03:57 PM
65611,05:48 AM,07:20 PM,01:04 AM,09:27 AM


In [ ]:
index_not_satisfy_format = df_ordinal_cols[df_ordinal_cols.applymap(lambda item: re.fullmatch(format, item) is None)].stack().index
index_not_satisfy_format

<ipython-input-17-c8ef9647d5b6>:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  index_not_satisfy_format = df_ordinal_cols[df_ordinal_cols.applymap(lambda item: re.fullmatch(format, item) is None)].stack().index


MultiIndex([(36985, 'moonrise_ord'),
            (25501, 'moonrise_ord'),
            (62647,  'moonset_ord'),
            (39776,  'moonset_ord'),
            (31084, 'moonrise_ord'),
            (54355, 'moonrise_ord'),
            (45475,  'moonset_ord'),
            (45510,  'moonset_ord'),
            (36978, 'moonrise_ord'),
            (25389, 'moonrise_ord'),
            ...
            (40035,  'moonset_ord'),
            (56577,  'moonset_ord'),
            ( 5901,  'moonset_ord'),
            (42614, 'moonrise_ord'),
            (62582,  'moonset_ord'),
            (36758, 'moonrise_ord'),
            (62759,  'moonset_ord'),
            (23073,  'moonset_ord'),
            ( 2967, 'moonrise_ord'),
            (51300,  'moonset_ord')],
           length=3512)

In [ ]:
df_ordinal_cols.stack()[index_not_satisfy_format].unique()

array(['No moonrise', 'No moonset'], dtype=object)

In [ ]:
# Get các giá trị ứng với hiện tượng có xảy ra
df_ordinal_cols_happen_stack = df_ordinal_cols.stack()
df_ordinal_cols_happen_stack = df_ordinal_cols_happen_stack[~df_ordinal_cols_happen_stack.index.isin(index_not_satisfy_format)]
len(df_ordinal_cols_happen_stack)

206656

In [ ]:
# Chuỗi nào có PM thì giá trị giờ cộng thêm 12 phút
def process_time(gold_time: str):
  if gold_time.endswith("PM"):
    parts = gold_time.split(":")
    hour = int(parts[0]) + 12
    gold_time = f"{hour}:{parts[1]}"

  res = re.split("(AM|PM)", gold_time)[0].strip()
  res = res.split(":")[0]
  return res

df_ordinal_cols_happen_stack = df_ordinal_cols_happen_stack.apply(lambda item: process_time(item))
df_ordinal_cols_happen_stack.unique()


array(['07', '18', '20', '15', '14', '06', '23', '10', '17', '05', '19',
       '01', '09', '16', '11', '03', '08', '24', '12', '22', '21', '02',
       '13', '04'], dtype=object)

In [ ]:
# Cập nhật
df_ordinal_cols_stack = df_ordinal_cols.stack()
df_ordinal_cols_stack[df_ordinal_cols_happen_stack.index] = df_ordinal_cols_happen_stack

df_ordinal_cols_stack.unstack()

,sunrise_ord,sunset_ord,moonrise_ord,moonset_ord
56032,07,18,07,20
34107,07,15,14,20
54115,06,18,23,10
9861,06,17,05,15
65611,05,19,01,09
...,...,...,...,...
19337,06,18,22,09
4582,05,18,06,19
23724,06,19,18,05
11085,06,17,10,23


In [ ]:
df[ordinal_cols] = df_ordinal_cols_stack.unstack()

df['moonrise_ord'].unique()

array(['07', '14', '23', '05', '01', '18', '20', '03', '08', '24',
       'No moonrise', '22', '21', '16', '11', '10', '09', '12', '04',
       '02', '06', '19', '13', '17', '15'], dtype=object)

## Kiểm tra nội dung các cột `target`


In [ ]:
print(f"{target_col} -> {df[target_col].unique().tolist()}")


temp_bin_target -> ['low', 'medium', 'high']


## Fill missing value


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="mean"), numeric_cols),
        ("numCat", SimpleImputer(strategy="most_frequent"), numericCat_cols),
        ("cat", SimpleImputer(strategy="most_frequent"), cat_cols),
        ("target", SimpleImputer(strategy="most_frequent"), [target_col]),
    ]
)

df = preprocessor.fit_transform(df)

df = pd.DataFrame(
    df, columns=numeric_cols + numericCat_cols + cat_cols + [target_col]
)

df.head()


,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,...,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,sunrise_ord,sunset_ord,moonrise_ord,moonset_ord,temp_bin_target
0,18.0,57.0,30.39,81.0,75.0,10.0,1.3,24.2,384.8,52.0,...,0.925,21.09,23.495,1.0,Waxing Crescent,07,18,07,20,low
1,10.1,247.0,30.55,80.0,2.0,10.0,0.3,17.0,362.6,26.0,...,0.925,17.575,29.23,37.0,Waxing Crescent,07,15,14,20,low
2,24.5,75.0,29.83,89.0,100.0,19.0,0.0,33.6,177.6,31.0,...,0.925,5.92,9.065,69.0,Waning Gibbous,06,18,23,10,medium
3,22.3,211.0,30.02,12.0,0.0,10.0,6.0,25.7,240.3,76.5,...,3.5,5.7,13.8,5.0,Waning Crescent,06,17,05,15,low
4,3.6,256.0,29.94,40.0,0.0,10.0,6.0,3.7,264.55,80.0,...,6.29,8.14,13.32,71.0,Waning Gibbous,05,19,01,09,medium


In [ ]:
np.all(df.isnull().sum() == 0)


True

## Chuyển đổi các cột về đúng kiểu dữ liệu


In [ ]:
df[numeric_cols] = df[numeric_cols].astype("float32")
df[numericCat_cols] = df[numericCat_cols].astype("float32")
df[cat_cols] = df[cat_cols].astype("category")
df[target_col] = df[target_col].astype("category")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52542 entries, 0 to 52541
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   wind_kph_num                      52542 non-null  float32 
 1   wind_degree_num                   52542 non-null  float32 
 2   pressure_in_num                   52542 non-null  float32 
 3   humidity_num                      52542 non-null  float32 
 4   cloud_num                         52542 non-null  float32 
 5   visibility_km_num                 52542 non-null  float32 
 6   uv_index_num                      52542 non-null  float32 
 7   gust_kph_num                      52542 non-null  float32 
 8   air_quality_Carbon_Monoxide_num   52542 non-null  float32 
 9   air_quality_Ozone_num             52542 non-null  float32 
 10  air_quality_Nitrogen_dioxide_num  52542 non-null  float32 
 11  air_quality_Sulphur_dioxide_num   52542 non-null  floa

## Loại bỏ duplicates


In [ ]:
print(f"Tỉ lệ duplicates: {len(df.index[df.duplicated()]) / len(df.index) * 100}")
df = df.drop_duplicates().reset_index(drop=True)

True

## Chuẩn bị thứ tự cho các cột `ordinal`, `binary`


In [ ]:
ordinal_cols

NameError: name 'ordinal_cols' is not defined

In [ ]:
a = df['moonset_ord'].unique().tolist()
a

['20',
 '10',
 '15',
 '09',
 '11',
 '05',
 '07',
 '16',
 '12',
 '24',
 '08',
 '22',
 '02',
 '23',
 '13',
 '03',
 '01',
 '06',
 '17',
 '19',
 '21',
 '04',
 '18',
 'No moonset',
 '14']

In [ ]:
FEATURE_ORDINAL_DICT = {
    "sunrise_ord": ["02", "03", "04", "05", "06", "07", "08", "09", "10", "11"],
    "sunset_ord": ['12', '14','15', '16', '17', '18',  '19',  '20', '21', '22', '23' ],
    "moonrise_ord": [
        "No moonrise",
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
    ],
    "moonset_ord": [
        'No moonset',
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
    ],
}

# dc2

## Đọc dữ liệu


In [ ]:
df = myfuncs.load_python_object("artifacts/data_ingestion/train_data.pkl")

df.head()


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,temp_bin
56032,Luxembourg,Luxembourg,49.6117,6.1300,Europe/Luxembourg,1740822300,2025-03-01 10:45,3.0,37.4,Partly cloudy,...,23.495,2,2,07:18 AM,06:18 PM,07:55 AM,08:30 PM,Waxing Crescent,1,low
34107,Sweden,Stockholm,59.3333,18.0500,Europe/Stockholm,1731056400,2024-11-08 10:00,7.3,45.1,Sunny,...,29.230,2,2,07:26 AM,03:37 PM,02:20 PM,08:48 PM,Waxing Crescent,37,low
54115,Palau,Airai,7.3575,134.5578,Pacific/Palau,1739960100,2025-02-19 19:15,27.3,81.1,Overcast,...,9.065,1,1,06:18 AM,06:13 PM,11:15 PM,10:29 AM,Waning Gibbous,69,medium
9861,South Africa,Pretoria,-25.7500,28.1900,Africa/Johannesburg,1720098900,2024-07-04 15:15,19.1,66.4,Sunny,...,13.800,1,1,06:54 AM,05:29 PM,05:33 AM,03:57 PM,Waning Crescent,5,low
65611,Macedonia,Skopje,42.0000,21.4333,Europe/Skopje,1745053200,2025-04-19 11:00,21.3,70.3,Partly Cloudy,...,13.320,1,1,05:48 AM,07:20 PM,01:04 AM,09:27 AM,Waning Gibbous,71,medium


Kich thuoc


In [ ]:
df.shape


(52542, 42)

## Ý nghĩa các cột

| Cot                       | Y nghia                                                               | Don vi | Phan loai |
| ------------------------- | --------------------------------------------------------------------- | ------ | --------- |
| **country**                     | Đất nước mà dữ liệu được đo                                     | none   | Nominal   |
| **location_name**                       | Tên thành phố                                                          | none   | Nominal   |
| **latitude**             | vĩ độ của thành phố                                                        | none   | ordinal   |
| **longitude**             | kinh độ của thành phố                                                        | none   | ordinal   |
| **timezone**             | Giờ khu vực                                                        | none   | ordinal   |
| **last_updated_epoch**            |  thời điểm cập nhật dữ liệu cuối cùng dưới dạng Unix timestamp.       | none   | ordinal   |
| **last_updated**            |  thời điểm cập nhật dữ liệu cuối cùng (local time)       | none   | ordinal   |
| **temperature_celsius**            |  nhiệt độ dưới dạng C       | none   | ordinal   |
| **temperature_fahrenheit**            |  nhiệt độ dưới dạng F      | none   | ordinal   |
| **condition_text**            |  Tình trạng thời tiết      | none   | ordinal   |
| **wind_mph**         | Tốc độ gió (mile / hour)      | none   | ordinal   |
| wind_kph         | Tốc độ gió (km / hour)      | none   | numeric   |
| wind_degree        | Hướng gió theo độ    | none   | numeric   |
| **wind_direction**         | Hướng gió chi tiết hơn     | none   | numeric   |
| **pressure_mb**         | Áp suất (milibars)    | none   | numeric   |
| pressure_in         | Áp suất (inches)    | none   | numeric   |
| **precip_mm**         | Lượng mưa (mm)    | none   | numeric   |
| **precip_in**         | Lượng mưa (inches)    | none   | numeric   |
| humidity         | Độ ẩm (%)    | none   | numeric   |
| cloud         | Phần trăm mây bao phủ (%)    | none   | numeric   |
| **feels_like_celsius**         | Nhiệt độ cơ thể cảm nhận được thay vì là thực tế    | none   | numeric   |
| **feels_like_fahrenheit**         | Nhiệt độ cơ thể cảm nhận được thay vì là thực tế    | none   | numeric   |
| visibility_km         | Tầm nhìn (km)    | none   | numeric   |
| **visibility_miles**         | Tầm nhìn (mile)    | none   | numeric   |
| uv_index         | Chỉ số tia UV    | none   | numeric   |
| **gust_mph**         | sự tăng tốc đột ngột và mạnh mẽ của gió (mile / h)    | none   | numeric   |
| gust_kph         | sự tăng tốc đột ngột và mạnh mẽ của gió (km / h)    | none   | numeric   |
| air_quality_Carbon_Monoxide         | Nồng độ $CO$    | none   | numeric   |
| air_quality_Ozone         | Nồng độ $O_3$    | none   | numeric   |
| air_quality_Nitrogen_dioxide         | Nồng độ $NO_2$    | none   | numeric   |
| air_quality_Sulphur_dioxide         | Nồng độ $SO_2$    | none   | numeric   |
| air_quality_PM2.5         | Nồng độ $PM2.5$    | none   | numeric   |
| air_quality_PM10         | Nồng độ $PM10$    | none   | numeric   |
| **air_quality_us-epa-index**         | Chỉ số AQI    | none   | numeric   |
| **air_quality_gb-defra-index**         | Chỉ số AQI    | none   | numeric   |
| sunrise         | Thời điểm mặt trời lên   | none   | ordinal   |
| sunset         | Thời điểm mặt trời lặn   | none   | ordinal   |
| moonrise         | Thời điểm trăng lên   | none   | ordinal   |
| moonset         | Thời điểm trăng lặn   | none   | ordinal   |
| moon_phase         | Pha của mặt trăng   | none   | nominal   |
| moon_illumination         | Độ sáng của mặt trăng (%)   | none   | numeric |


## Xóa các cột không cần thiết




### Xóa các cột

In [ ]:
df.columns

Index(['country', 'location_name', 'latitude', 'longitude', 'timezone',
       'last_updated_epoch', 'last_updated', 'temperature_celsius',
       'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph',
       'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in',
       'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius',
       'feels_like_fahrenheit', 'visibility_km', 'visibility_miles',
       'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise',
       'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination',
       'temp_bin'],
      dtype='object')

In [ ]:
df = df.drop(
    columns=[
        "country",
        "location_name",
        "latitude",
        "longitude",
        "timezone",
        "last_updated_epoch",
        "last_updated",
        "temperature_celsius",
        "temperature_fahrenheit",
        "condition_text",
        "wind_mph",
        "wind_direction",
        "pressure_mb",
        "precip_mm",
        "precip_in",
        "feels_like_celsius",
        "feels_like_fahrenheit",
        "visibility_miles",
        "gust_mph",
        "air_quality_us-epa-index",
        "air_quality_gb-defra-index",
        'sunrise',
        'sunset',
        'moonrise',
        'moonset'
    ]
)

df.columns

Index(['wind_kph', 'wind_degree', 'pressure_in', 'humidity', 'cloud',
       'visibility_km', 'uv_index', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'moon_phase', 'moon_illumination', 'temp_bin'],
      dtype='object')

### Tỉ lệ missing các cột

In [ ]:
null_percent = df.isnull().mean() * 100
null_percent = null_percent.sort_values(ascending=False)
null_percent


,0
wind_kph,0.0
air_quality_Ozone,0.0
moon_illumination,0.0
moon_phase,0.0
air_quality_PM10,0.0
air_quality_PM2.5,0.0
air_quality_Sulphur_dioxide,0.0
air_quality_Nitrogen_dioxide,0.0
air_quality_Carbon_Monoxide,0.0
wind_degree,0.0


### Xóa các cột có tỉ lệ missing lớn

Ti le missing của các cột đều = 0-> Khong xoa cot nao het !


In [ ]:
df.shape


(52542, 21)

## Đổi tên cột

In [ ]:
rename_dict = {
    "wind_kph": "wind_kph_num",
    "wind_degree": "wind_degree_num",
    "pressure_in": "pressure_in_num",
    "humidity": "humidity_num",
    "cloud": "cloud_num",
    "visibility_km": "visibility_km_num",
    "uv_index": "uv_index_num",
    "gust_kph": "gust_kph_num",
    "air_quality_Carbon_Monoxide": "air_quality_Carbon_Monoxide_num",
    "air_quality_Ozone": "air_quality_Ozone_num",
    "air_quality_Nitrogen_dioxide": "air_quality_Nitrogen_dioxide_num",
    "air_quality_Sulphur_dioxide": "air_quality_Sulphur_dioxide_num",
    "air_quality_PM2.5": "air_quality_PM2_5_num",
    "air_quality_PM10": "air_quality_PM10_num",
    "moon_phase": "moon_phase_nom",
    "moon_illumination": "moon_illumination_num",
    "temp_bin": "temp_bin_target",

}


df = df.rename(columns=rename_dict)

df.columns


Index(['wind_kph_num', 'wind_degree_num', 'pressure_in_num', 'humidity_num',
       'cloud_num', 'visibility_km_num', 'uv_index_num', 'gust_kph_num',
       'air_quality_Carbon_Monoxide_num', 'air_quality_Ozone_num',
       'air_quality_Nitrogen_dioxide_num', 'air_quality_Sulphur_dioxide_num',
       'air_quality_PM2_5_num', 'air_quality_PM10_num', 'moon_phase_nom',
       'moon_illumination_num', 'temp_bin_target'],
      dtype='object')

## Sắp xếp các cột theo đúng thứ tự

In [ ]:
numeric_cols, numericCat_cols, cat_cols, binary_cols, nominal_cols, ordinal_cols, target_col = myfuncs.get_different_types_cols_from_df_4(df)


df = df[
    numeric_cols
    + numericCat_cols
    + binary_cols
    + nominal_cols
    + ordinal_cols
    + [target_col]
]


df.head()


,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,air_quality_Nitrogen_dioxide_num,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,temp_bin_target
56032,18.0,57,30.39,81,75,10.0,1.3,24.2,384.80,52.0,17.205,0.925,21.090,23.495,1,Waxing Crescent,low
34107,10.1,247,30.55,80,2,10.0,0.3,17.0,362.60,26.0,31.820,0.925,17.575,29.230,37,Waxing Crescent,low
54115,24.5,75,29.83,89,100,19.0,0.0,33.6,177.60,31.0,0.925,0.925,5.920,9.065,69,Waning Gibbous,medium
9861,22.3,211,30.02,12,0,10.0,6.0,25.7,240.30,76.5,1.100,3.500,5.700,13.800,5,Waning Crescent,low
65611,3.6,256,29.94,40,0,10.0,6.0,3.7,264.55,80.0,8.695,6.290,8.140,13.320,71,Waning Gibbous,medium


## Kiểm tra kiểu dữ liệu các cột

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 52542 entries, 56032 to 4001
Data columns (total 17 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   wind_kph_num                      52542 non-null  float64 
 1   wind_degree_num                   52542 non-null  int64   
 2   pressure_in_num                   52542 non-null  float64 
 3   humidity_num                      52542 non-null  int64   
 4   cloud_num                         52542 non-null  int64   
 5   visibility_km_num                 52542 non-null  float64 
 6   uv_index_num                      52542 non-null  float64 
 7   gust_kph_num                      52542 non-null  float64 
 8   air_quality_Carbon_Monoxide_num   52542 non-null  float64 
 9   air_quality_Ozone_num             52542 non-null  float64 
 10  air_quality_Nitrogen_dioxide_num  52542 non-null  float64 
 11  air_quality_Sulphur_dioxide_num   52542 non-null  float6

SAI:

- cac cot nominal, ordinal


### Chuyển kdl = kdl mong muốn + NAN


In [ ]:
for col in df.columns.tolist()[15:]:
  print(f"{col} -> {set(map(type, df[col]))}")


moon_phase_nom -> {<class 'str'>}
temp_bin_target -> {<class 'str'>}


- Tất cả các cột đều đúng kiểu dữ liệu


## Kiểm tra nội dung các cột `binary`


In [ ]:
for col in binary_cols:
  print(f"{col} -> {df[col].unique().tolist()}")

Không có cột nào hết


## Kiểm tra nội dung các cột `nominal`


In [ ]:
for col in nominal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


moon_phase_nom -> ['Waxing Crescent', 'Waning Gibbous', 'Waning Crescent', 'Last Quarter', 'Waxing Gibbous', 'First Quarter', 'Full Moon', 'New Moon']


## Kiểm tra nội dung các cột `ordinal`


In [ ]:
for col in ordinal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


## Kiểm tra nội dung các cột `target`


In [ ]:
print(f"{target_col} -> {df[target_col].unique().tolist()}")


temp_bin_target -> ['low', 'medium', 'high']


## Fill missing value


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="mean"), numeric_cols),
        ("numCat", SimpleImputer(strategy="most_frequent"), numericCat_cols),
        ("cat", SimpleImputer(strategy="most_frequent"), cat_cols),
        ("target", SimpleImputer(strategy="most_frequent"), [target_col]),
    ]
)

df = preprocessor.fit_transform(df)

df = pd.DataFrame(
    df, columns=numeric_cols + numericCat_cols + cat_cols + [target_col]
)

df.head()


,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,air_quality_Nitrogen_dioxide_num,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,temp_bin_target
0,18.0,57.0,30.39,81.0,75.0,10.0,1.3,24.2,384.8,52.0,17.205,0.925,21.09,23.495,1.0,Waxing Crescent,low
1,10.1,247.0,30.55,80.0,2.0,10.0,0.3,17.0,362.6,26.0,31.82,0.925,17.575,29.23,37.0,Waxing Crescent,low
2,24.5,75.0,29.83,89.0,100.0,19.0,0.0,33.6,177.6,31.0,0.925,0.925,5.92,9.065,69.0,Waning Gibbous,medium
3,22.3,211.0,30.02,12.0,0.0,10.0,6.0,25.7,240.3,76.5,1.1,3.5,5.7,13.8,5.0,Waning Crescent,low
4,3.6,256.0,29.94,40.0,0.0,10.0,6.0,3.7,264.55,80.0,8.695,6.29,8.14,13.32,71.0,Waning Gibbous,medium


In [ ]:
np.all(df.isnull().sum() == 0)


True

## Chuyển đổi các cột về đúng kiểu dữ liệu


In [ ]:
df[numeric_cols] = df[numeric_cols].astype("float32")
df[numericCat_cols] = df[numericCat_cols].astype("float32")
df[cat_cols] = df[cat_cols].astype("category")
df[target_col] = df[target_col].astype("category")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52542 entries, 0 to 52541
Data columns (total 17 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   wind_kph_num                      52542 non-null  float32 
 1   wind_degree_num                   52542 non-null  float32 
 2   pressure_in_num                   52542 non-null  float32 
 3   humidity_num                      52542 non-null  float32 
 4   cloud_num                         52542 non-null  float32 
 5   visibility_km_num                 52542 non-null  float32 
 6   uv_index_num                      52542 non-null  float32 
 7   gust_kph_num                      52542 non-null  float32 
 8   air_quality_Carbon_Monoxide_num   52542 non-null  float32 
 9   air_quality_Ozone_num             52542 non-null  float32 
 10  air_quality_Nitrogen_dioxide_num  52542 non-null  float32 
 11  air_quality_Sulphur_dioxide_num   52542 non-null  floa

## Loại bỏ duplicates


In [ ]:
print(f"Tỉ lệ duplicates: {len(df.index[df.duplicated()]) / len(df.index) * 100}" )
df = df.drop_duplicates().reset_index()

0.41109969167523125

## FEATURE_ORDINAL_DICT


In [ ]:
binary_cols + ordinal_cols

[]

In [ ]:
FEATURE_ORDINAL_DICT = {

}

# r

In [4]:
df = myfuncs.load_python_object("artifacts/data_correction_dc2/data.pkl")
transformer = myfuncs.load_python_object("artifacts/data_correction_dc2/transformer.pkl")
feature_ordinal_dict = myfuncs.load_python_object("artifacts/data_correction_dc2/feature_ordinal_dict.pkl")
df.head()

,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,air_quality_Nitrogen_dioxide_num,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,temp_bin_target
0,18.000000,57.0,30.389999,81.0,75.0,10.0,1.3,24.200001,384.799988,52.0,17.205,0.925,21.090000,23.495001,1.0,Waxing Crescent,low
1,10.100000,247.0,30.549999,80.0,2.0,10.0,0.3,17.000000,362.600006,26.0,31.820,0.925,17.575001,29.230000,37.0,Waxing Crescent,low
2,24.500000,75.0,29.830000,89.0,100.0,19.0,0.0,33.599998,177.600006,31.0,0.925,0.925,5.920000,9.065000,69.0,Waning Gibbous,medium
3,22.299999,211.0,30.020000,12.0,0.0,10.0,6.0,25.700001,240.300003,76.5,1.100,3.500,5.700000,13.800000,5.0,Waning Crescent,low
4,3.600000,256.0,29.940001,40.0,0.0,10.0,6.0,3.700000,264.549988,80.0,8.695,6.290,8.140000,13.320000,71.0,Waning Gibbous,medium


In [5]:
numeric_cols = myfuncs.get_different_types_cols_from_df_4(df)[0]
numericcat_cols = myfuncs.get_different_types_cols_from_df_4(df)[1]

df[numeric_cols].describe().loc[['min', 'max']].T

,min,max
wind_kph_num,3.600000,2963.199951
wind_degree_num,1.000000,360.000000
pressure_in_num,27.959999,88.589996
humidity_num,2.000000,100.000000
cloud_num,0.000000,100.000000
visibility_km_num,0.000000,32.000000
uv_index_num,0.000000,16.299999
gust_kph_num,3.600000,2970.399902
air_quality_Carbon_Monoxide_num,-9999.000000,38879.398438
air_quality_Ozone_num,0.000000,480.700012


cột air_quality_Carbon_Monoxide_num có giá trị min < 0

cột air_quality_Sulphur_dioxide_num có giá trị min < 0

Fill các giá trị < 0 của 2 cột bằng median

In [7]:
col_names = ["air_quality_Carbon_Monoxide_num", "air_quality_Sulphur_dioxide_num"]

for col in col_names:
  df[col][df.index[df[col] < 0]] = np.percentile(df[col], 50)

df[numeric_cols].describe().loc[['min', 'max']].T

<ipython-input-7-2b864edab658>:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col][df.index[df[col] < 0]] = np.percentile(df[col], 50)
<ipython-input-7-2b864edab658>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

,min,max
wind_kph_num,3.600000,2963.199951
wind_degree_num,1.000000,360.000000
pressure_in_num,27.959999,88.589996
humidity_num,2.000000,100.000000
cloud_num,0.000000,100.000000
visibility_km_num,0.000000,32.000000
uv_index_num,0.000000,16.299999
gust_kph_num,3.600000,2970.399902
air_quality_Carbon_Monoxide_num,81.000000,38879.398438
air_quality_Ozone_num,0.000000,480.700012


In [10]:
subplot_titles = sum(([item, "", ""] for item in numeric_cols), [])

fig = make_subplots(rows=len(numeric_cols), cols=3, subplot_titles=subplot_titles)

for row, col in enumerate(numeric_cols, 1):
    data = df[col]

    # Vẽ Histogram
    fig_hist = px.histogram(
        x=data,
        nbins=100,
    )

    # Thêm đường viền cho các cột histogram
    fig_hist.update_traces(marker=dict(line=dict(width=1, color="black")))

    # Vẽ box plot
    fig_box = px.box(
        y=data,
    )

    # Vẽ violin plot
    fig_violin = px.violin(
        y=data,
        box=True,
    )

    fig.add_trace(fig_hist.data[0], row=row, col=1)
    fig.add_trace(fig_box.data[0], row=row, col=2)
    fig.add_trace(fig_violin.data[0], row=row, col=3)

fig.update_layout(
    width=400 * 3,  # Độ rộng biểu đồ (px)
    height=len(numeric_cols) * 500,  # Độ cao biểu đồ (px)
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

cột wind_kph_num bị lệch trên

cột wind_degree_num đều

cột pressure_in_num bị lệch trên

cột humidity_num đều

cột cloud_num đều

cột visibility_km_num bị lệch 2 bên

cột uv_index_num đều

cột gust_kph_num bị lệch trên

cột air_quality_Carbon_Monoxide_num  bị lệch trên

cột air_quality_Ozone_num bị lệch trên

cột air_quality_Nitrogen_dioxide_num  bị lệch trên

cột air_quality_Sulphur_dioxide_num bị lệch trên

cột air_quality_PM2_5_num bị lệch trên

cột air_quality_PM10_num bị lệch trên

cột moon_illumination_num đều

Fill các giá trị outliers của các cột wind_kph_num, pressure_in_num, visibility_km_num, gust_kph_num, air_quality_Carbon_Monoxide_num, air_quality_Ozone_num, air_quality_Nitrogen_dioxide_num, air_quality_Sulphur_dioxide_num, air_quality_PM2_5_num, air_quality_PM10_num bằng median của cột

In [11]:
col_names = [
    'wind_kph_num',
    'pressure_in_num',
    'visibility_km_num',
    'gust_kph_num',
    'air_quality_Carbon_Monoxide_num',
    'air_quality_Ozone_num',
    'air_quality_Nitrogen_dioxide_num',
    'air_quality_Sulphur_dioxide_num',
    'air_quality_PM2_5_num',
    'air_quality_PM10_num',
]

for col_name in col_names:
  df[col_name] = myfuncs.replace_outliers_with_new_value_34(df[col_name], np.percentile(df[col_name], 50))

/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1607: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [12]:
subplot_titles = sum(([item, "", ""] for item in numeric_cols), [])

fig = make_subplots(rows=len(numeric_cols), cols=3, subplot_titles=subplot_titles)

for row, col in enumerate(numeric_cols, 1):
    data = df[col]

    # Vẽ Histogram
    fig_hist = px.histogram(
        x=data,
        nbins=100,
    )

    # Thêm đường viền cho các cột histogram
    fig_hist.update_traces(marker=dict(line=dict(width=1, color="black")))

    # Vẽ box plot
    fig_box = px.box(
        y=data,
    )

    # Vẽ violin plot
    fig_violin = px.violin(
        y=data,
        box=True,
    )

    fig.add_trace(fig_hist.data[0], row=row, col=1)
    fig.add_trace(fig_box.data[0], row=row, col=2)
    fig.add_trace(fig_violin.data[0], row=row, col=3)

fig.update_layout(
    width=400 * 3,  # Độ rộng biểu đồ (px)
    height=len(numeric_cols) * 500,  # Độ cao biểu đồ (px)
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [13]:
for col in numericcat_cols:
  print(f"{col} -> {df[col].unique().tolist()}")

In [17]:
df.shape

(52326, 17)

In [16]:
# Lưu dữ liệu
folder = "artifacts/data_correction_dc3"
os.makedirs(folder, exist_ok=True)

myfuncs.save_python_object(os.path.join(folder, "data.pkl"), df)
myfuncs.save_python_object(os.path.join(folder, "transformer.pkl"), transformer)
myfuncs.save_python_object(os.path.join(folder, "feature_ordinal_dict.pkl"), feature_ordinal_dict)

# r

In [18]:
df = myfuncs.load_python_object("artifacts/data_correction_dc1/data.pkl")

df.head()

,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,...,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,sunrise_ord,sunset_ord,moonrise_ord,moonset_ord,temp_bin_target
0,18.000000,57.0,30.389999,81.0,75.0,10.0,1.3,24.200001,384.799988,52.0,...,0.925,21.090000,23.495001,1.0,Waxing Crescent,07,18,07,20,low
1,10.100000,247.0,30.549999,80.0,2.0,10.0,0.3,17.000000,362.600006,26.0,...,0.925,17.575001,29.230000,37.0,Waxing Crescent,07,15,14,20,low
2,24.500000,75.0,29.830000,89.0,100.0,19.0,0.0,33.599998,177.600006,31.0,...,0.925,5.920000,9.065000,69.0,Waning Gibbous,06,18,23,10,medium
3,22.299999,211.0,30.020000,12.0,0.0,10.0,6.0,25.700001,240.300003,76.5,...,3.500,5.700000,13.800000,5.0,Waning Crescent,06,17,05,15,low
4,3.600000,256.0,29.940001,40.0,0.0,10.0,6.0,3.700000,264.549988,80.0,...,6.290,8.140000,13.320000,71.0,Waning Gibbous,05,19,01,09,medium


| Cot                       | Y nghia                                                               | Don vi | Phan loai |
| ------------------------- | --------------------------------------------------------------------- | ------ | --------- |
| wind_kph         | Tốc độ gió (km / hour)      | none   | numeric   |
| wind_degree        | Hướng gió theo độ    | none   | numeric   |
| pressure_in         | Áp suất (inches)    | none   | numeric   |
| humidity         | Độ ẩm (%)    | none   | numeric   |
| cloud         | Phần trăm mây bao phủ (%)    | none   | numeric   |
| visibility_km         | Tầm nhìn (km)    | none   | numeric   |
| uv_index         | Chỉ số tia UV    | none   | numeric   |
| gust_kph         | sự tăng tốc đột ngột và mạnh mẽ của gió (km / h)    | none   | numeric   |
| air_quality_Carbon_Monoxide         | Nồng độ $CO$    | none   | numeric   |
| air_quality_Ozone         | Nồng độ $O_3$    | none   | numeric   |
| air_quality_Nitrogen_dioxide         | Nồng độ $NO_2$    | none   | numeric   |
| air_quality_Sulphur_dioxide         | Nồng độ $SO_2$    | none   | numeric   |
| air_quality_PM2.5         | Nồng độ $PM2.5$    | none   | numeric   |
| air_quality_PM10         | Nồng độ $PM10$    | none   | numeric   |
| sunrise         | Thời điểm mặt trời lên   | none   | ordinal   |
| sunset         | Thời điểm mặt trời lặn   | none   | ordinal   |
| moonrise         | Thời điểm trăng lên   | none   | ordinal   |
| moonset         | Thời điểm trăng lặn   | none   | ordinal   |
| moon_phase         | Pha của mặt trăng   | none   | nominal   |
| moon_illumination         | Độ sáng của mặt trăng (%)   | none   | numeric |


In [20]:
df.columns

Index(['wind_kph_num', 'wind_degree_num', 'pressure_in_num', 'humidity_num',
       'cloud_num', 'visibility_km_num', 'uv_index_num', 'gust_kph_num',
       'air_quality_Carbon_Monoxide_num', 'air_quality_Ozone_num',
       'air_quality_Nitrogen_dioxide_num', 'air_quality_Sulphur_dioxide_num',
       'air_quality_PM2_5_num', 'air_quality_PM10_num',
       'moon_illumination_num', 'moon_phase_nom', 'sunrise_ord', 'sunset_ord',
       'moonrise_ord', 'moonset_ord', 'temp_bin_target'],
      dtype='object')

Bỏ 2 cột sunset, moonset

In [21]:
df = df.drop(columns = ['sunset_ord', 'moonset_ord'])

In [22]:
ordinal_cols = myfuncs.get_different_types_cols_from_df_4(df)[5]
ordinal_cols

['sunrise_ord', 'moonrise_ord']

In [23]:
for col in ordinal_cols:
  print(f"{col} -> {df[col].unique().tolist()}")

sunrise_ord -> ['07', '06', '05', '04', '08', '03', '09', '10', '11', '02']
moonrise_ord -> ['07', '14', '23', '05', '01', '18', '20', '03', '08', '24', 'No moonrise', '22', '21', '16', '11', '10', '09', '12', '04', '02', '06', '19', '13', '17', '15']


In [25]:
col_name = 'sunrise_ord'
replace_value = [
    [['02', '03', '04', '05'], 'early_morning'],
    [['06','07', '08', '09' ], 'morning'],
    [['10', '11'], 'midday']
]
df[col_name] = myfuncs.replace_in_category_series_33(df[col_name], replace_value)
df[col_name].unique()

['morning', 'early_morning', 'midday']
Categories (3, string): [early_morning, midday, morning]

In [28]:
a = df['moonrise_ord'].unique().tolist()
a

['07',
 '14',
 '23',
 '05',
 '01',
 '18',
 '20',
 '03',
 '08',
 '24',
 'No moonrise',
 '22',
 '21',
 '16',
 '11',
 '10',
 '09',
 '12',
 '04',
 '02',
 '06',
 '19',
 '13',
 '17',
 '15']

In [29]:
col_name = 'moonrise_ord'
replace_value = [
    [['02','03', '04', '05'], 'early_morning'],
    [['06','07', '08', '09'], 'morning'],
    [['10','11', '12', '13'], 'midday'],
    [['14','15', '16', '17'], 'afternoon'],
    [['18','19', '20', '21', '22'], 'evening'],
    [['23','24', '01'], 'midnight'],
]
df[col_name] = myfuncs.replace_in_category_series_33(df[col_name], replace_value)
df[col_name].unique()

['morning', 'afternoon', 'midnight', 'early_morning', 'evening', 'No moonrise', 'midday']
Categories (7, string): [No moonrise, afternoon, early_morning, evening, midday, midnight, morning]

In [30]:
numeric_cols = myfuncs.get_different_types_cols_from_df_4(df)[0]

In [31]:
df[numeric_cols].describe().loc[['min', 'max']].T

,min,max
wind_kph_num,3.600000,2963.199951
wind_degree_num,1.000000,360.000000
pressure_in_num,27.959999,88.589996
humidity_num,2.000000,100.000000
cloud_num,0.000000,100.000000
visibility_km_num,0.000000,32.000000
uv_index_num,0.000000,16.299999
gust_kph_num,3.600000,2970.399902
air_quality_Carbon_Monoxide_num,-9999.000000,38879.398438
air_quality_Ozone_num,0.000000,480.700012


Fill các giá trị < 0 của 2 cột air_quality_Carbon_Monoxide_num và air_quality_Sulphur_dioxide_num bằng median

In [32]:
col_names = ["air_quality_Carbon_Monoxide_num", "air_quality_Sulphur_dioxide_num"]

for col in col_names:
  df.loc[df.index[df[col] < 0], col] = np.percentile(df[col], 50)

df[numeric_cols].describe().loc[['min', 'max']].T

,min,max
wind_kph_num,3.600000,2963.199951
wind_degree_num,1.000000,360.000000
pressure_in_num,27.959999,88.589996
humidity_num,2.000000,100.000000
cloud_num,0.000000,100.000000
visibility_km_num,0.000000,32.000000
uv_index_num,0.000000,16.299999
gust_kph_num,3.600000,2970.399902
air_quality_Carbon_Monoxide_num,81.000000,38879.398438
air_quality_Ozone_num,0.000000,480.700012


Fill các giá trị outliers của các cột wind_kph_num, pressure_in_num, visibility_km_num, gust_kph_num, air_quality_Carbon_Monoxide_num, air_quality_Ozone_num, air_quality_Nitrogen_dioxide_num, air_quality_Sulphur_dioxide_num, air_quality_PM2_5_num, air_quality_PM10_num bằng median của cột

In [33]:
col_names = [
    'wind_kph_num',
    'pressure_in_num',
    'visibility_km_num',
    'gust_kph_num',
    'air_quality_Carbon_Monoxide_num',
    'air_quality_Ozone_num',
    'air_quality_Nitrogen_dioxide_num',
    'air_quality_Sulphur_dioxide_num',
    'air_quality_PM2_5_num',
    'air_quality_PM10_num',
]

for col_name in col_names:
  df[col_name] = myfuncs.replace_outliers_with_new_value_34(df[col_name], np.percentile(df[col_name], 50))

/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1607: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



# dc4

## Đọc dữ liệu


In [69]:
df = myfuncs.load_python_object("artifacts/data_ingestion/train_data.pkl")

df.head()


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,temp_bin
56032,Luxembourg,Luxembourg,49.6117,6.1300,Europe/Luxembourg,1740822300,2025-03-01 10:45,3.0,37.4,Partly cloudy,...,23.495,2,2,07:18 AM,06:18 PM,07:55 AM,08:30 PM,Waxing Crescent,1,low
34107,Sweden,Stockholm,59.3333,18.0500,Europe/Stockholm,1731056400,2024-11-08 10:00,7.3,45.1,Sunny,...,29.230,2,2,07:26 AM,03:37 PM,02:20 PM,08:48 PM,Waxing Crescent,37,low
54115,Palau,Airai,7.3575,134.5578,Pacific/Palau,1739960100,2025-02-19 19:15,27.3,81.1,Overcast,...,9.065,1,1,06:18 AM,06:13 PM,11:15 PM,10:29 AM,Waning Gibbous,69,medium
9861,South Africa,Pretoria,-25.7500,28.1900,Africa/Johannesburg,1720098900,2024-07-04 15:15,19.1,66.4,Sunny,...,13.800,1,1,06:54 AM,05:29 PM,05:33 AM,03:57 PM,Waning Crescent,5,low
65611,Macedonia,Skopje,42.0000,21.4333,Europe/Skopje,1745053200,2025-04-19 11:00,21.3,70.3,Partly Cloudy,...,13.320,1,1,05:48 AM,07:20 PM,01:04 AM,09:27 AM,Waning Gibbous,71,medium


## Ý nghĩa các cột

| Cot                       | Y nghia                                                               | Don vi | Phan loai |
| ------------------------- | --------------------------------------------------------------------- | ------ | --------- |
| **country**                     | Đất nước mà dữ liệu được đo                                     | none   | Nominal   |
| **location_name**                       | Tên thành phố                                                          | none   | Nominal   |
| **latitude**             | vĩ độ của thành phố                                                        | none   | ordinal   |
| **longitude**             | kinh độ của thành phố                                                        | none   | ordinal   |
| **timezone**             | Giờ khu vực                                                        | none   | ordinal   |
| **last_updated_epoch**            |  thời điểm cập nhật dữ liệu cuối cùng dưới dạng Unix timestamp.       | none   | ordinal   |
| **last_updated**            |  thời điểm cập nhật dữ liệu cuối cùng (local time)       | none   | ordinal   |
| **temperature_celsius**            |  nhiệt độ dưới dạng C       | none   | ordinal   |
| **temperature_fahrenheit**            |  nhiệt độ dưới dạng F      | none   | ordinal   |
| **condition_text**            |  Tình trạng thời tiết      | none   | ordinal   |
| **wind_mph**         | Tốc độ gió (mile / hour)      | none   | ordinal   |
| wind_kph         | Tốc độ gió (km / hour)      | none   | numeric   |
| wind_degree        | Hướng gió theo độ    | none   | numeric   |
| **wind_direction**         | Hướng gió chi tiết hơn     | none   | numeric   |
| **pressure_mb**         | Áp suất (milibars)    | none   | numeric   |
| pressure_in         | Áp suất (inches)    | none   | numeric   |
| **precip_mm**         | Lượng mưa (mm)    | none   | numeric   |
| **precip_in**         | Lượng mưa (inches)    | none   | numeric   |
| humidity         | Độ ẩm (%)    | none   | numeric   |
| cloud         | Phần trăm mây bao phủ (%)    | none   | numeric   |
| **feels_like_celsius**         | Nhiệt độ cơ thể cảm nhận được thay vì là thực tế    | none   | numeric   |
| **feels_like_fahrenheit**         | Nhiệt độ cơ thể cảm nhận được thay vì là thực tế    | none   | numeric   |
| visibility_km         | Tầm nhìn (km)    | none   | numeric   |
| **visibility_miles**         | Tầm nhìn (mile)    | none   | numeric   |
| uv_index         | Chỉ số tia UV    | none   | numeric   |
| **gust_mph**         | sự tăng tốc đột ngột và mạnh mẽ của gió (mile / h)    | none   | numeric   |
| gust_kph         | sự tăng tốc đột ngột và mạnh mẽ của gió (km / h)    | none   | numeric   |
| air_quality_Carbon_Monoxide         | Nồng độ $CO$    | none   | numeric   |
| air_quality_Ozone         | Nồng độ $O_3$    | none   | numeric   |
| air_quality_Nitrogen_dioxide         | Nồng độ $NO_2$    | none   | numeric   |
| air_quality_Sulphur_dioxide         | Nồng độ $SO_2$    | none   | numeric   |
| air_quality_PM2.5         | Nồng độ $PM2.5$    | none   | numeric   |
| air_quality_PM10         | Nồng độ $PM10$    | none   | numeric   |
| **air_quality_us-epa-index**         | Chỉ số AQI    | none   | numeric   |
| **air_quality_gb-defra-index**         | Chỉ số AQI    | none   | numeric   |
| sunrise         | Thời điểm mặt trời lên   | none   | ordinal   |
| sunset         | Thời điểm mặt trời lặn   | none   | ordinal   |
| moonrise         | Thời điểm trăng lên   | none   | ordinal   |
| moonset         | Thời điểm trăng lặn   | none   | ordinal   |
| moon_phase         | Pha của mặt trăng   | none   | nominal   |
| moon_illumination         | Độ sáng của mặt trăng (%)   | none   | numeric |


## Xóa các cột không cần thiết




### Xóa các cột

In [70]:
df = df.drop(
    columns=[
        "country",
        "location_name",
        "latitude",
        "longitude",
        "timezone",
        "last_updated_epoch",
        "last_updated",
        "temperature_celsius",
        "temperature_fahrenheit",
        "condition_text",
        "wind_mph",
        "wind_direction",
        "pressure_mb",
        "precip_mm",
        "precip_in",
        "feels_like_celsius",
        "feels_like_fahrenheit",
        "visibility_miles",
        "gust_mph",
        "air_quality_us-epa-index",
        "air_quality_gb-defra-index",
        'sunset',
        'moonset'
    ]
)

df.columns

Index(['wind_kph', 'wind_degree', 'pressure_in', 'humidity', 'cloud',
       'visibility_km', 'uv_index', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'sunrise', 'moonrise', 'moon_phase', 'moon_illumination', 'temp_bin'],
      dtype='object')

### Tỉ lệ missing các cột

In [71]:
null_percent = df.isnull().mean() * 100
null_percent = null_percent.sort_values(ascending=False)
null_percent


,0
wind_kph,0.0
air_quality_Nitrogen_dioxide,0.0
moon_illumination,0.0
moon_phase,0.0
moonrise,0.0
sunrise,0.0
air_quality_PM10,0.0
air_quality_PM2.5,0.0
air_quality_Sulphur_dioxide,0.0
air_quality_Ozone,0.0


### Xóa các cột có tỉ lệ missing lớn

Ti le missing của các cột đều = 0-> Khong xoa cot nao het !


## Đổi tên cột

In [72]:
rename_dict = {
    "wind_kph": "wind_kph_num",
    "wind_degree": "wind_degree_num",
    "pressure_in": "pressure_in_num",
    "humidity": "humidity_num",
    "cloud": "cloud_num",
    "visibility_km": "visibility_km_num",
    "uv_index": "uv_index_num",
    "gust_kph": "gust_kph_num",
    "air_quality_Carbon_Monoxide": "air_quality_Carbon_Monoxide_num",
    "air_quality_Ozone": "air_quality_Ozone_num",
    "air_quality_Nitrogen_dioxide": "air_quality_Nitrogen_dioxide_num",
    "air_quality_Sulphur_dioxide": "air_quality_Sulphur_dioxide_num",
    "air_quality_PM2.5": "air_quality_PM2_5_num",
    "air_quality_PM10": "air_quality_PM10_num",
    "sunrise": "sunrise_ord",
    "moonrise": "moonrise_ord",
    "moon_phase": "moon_phase_nom",
    "moon_illumination": "moon_illumination_num",
    "temp_bin": "temp_bin_target",

}


df = df.rename(columns=rename_dict)

df.columns


Index(['wind_kph_num', 'wind_degree_num', 'pressure_in_num', 'humidity_num',
       'cloud_num', 'visibility_km_num', 'uv_index_num', 'gust_kph_num',
       'air_quality_Carbon_Monoxide_num', 'air_quality_Ozone_num',
       'air_quality_Nitrogen_dioxide_num', 'air_quality_Sulphur_dioxide_num',
       'air_quality_PM2_5_num', 'air_quality_PM10_num', 'sunrise_ord',
       'moonrise_ord', 'moon_phase_nom', 'moon_illumination_num',
       'temp_bin_target'],
      dtype='object')

## Sắp xếp các cột theo đúng thứ tự

In [73]:
numeric_cols, numericCat_cols, cat_cols, binary_cols, nominal_cols, ordinal_cols, target_col = myfuncs.get_different_types_cols_from_df_4(df)


df = df[
    numeric_cols
    + numericCat_cols
    + binary_cols
    + nominal_cols
    + ordinal_cols
    + [target_col]
]


df.head()


,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,air_quality_Nitrogen_dioxide_num,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,sunrise_ord,moonrise_ord,temp_bin_target
56032,18.0,57,30.39,81,75,10.0,1.3,24.2,384.80,52.0,17.205,0.925,21.090,23.495,1,Waxing Crescent,07:18 AM,07:55 AM,low
34107,10.1,247,30.55,80,2,10.0,0.3,17.0,362.60,26.0,31.820,0.925,17.575,29.230,37,Waxing Crescent,07:26 AM,02:20 PM,low
54115,24.5,75,29.83,89,100,19.0,0.0,33.6,177.60,31.0,0.925,0.925,5.920,9.065,69,Waning Gibbous,06:18 AM,11:15 PM,medium
9861,22.3,211,30.02,12,0,10.0,6.0,25.7,240.30,76.5,1.100,3.500,5.700,13.800,5,Waning Crescent,06:54 AM,05:33 AM,low
65611,3.6,256,29.94,40,0,10.0,6.0,3.7,264.55,80.0,8.695,6.290,8.140,13.320,71,Waning Gibbous,05:48 AM,01:04 AM,medium


## Kiểm tra kiểu dữ liệu các cột

In [74]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 52542 entries, 56032 to 4001
Data columns (total 19 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   wind_kph_num                      52542 non-null  float64 
 1   wind_degree_num                   52542 non-null  int64   
 2   pressure_in_num                   52542 non-null  float64 
 3   humidity_num                      52542 non-null  int64   
 4   cloud_num                         52542 non-null  int64   
 5   visibility_km_num                 52542 non-null  float64 
 6   uv_index_num                      52542 non-null  float64 
 7   gust_kph_num                      52542 non-null  float64 
 8   air_quality_Carbon_Monoxide_num   52542 non-null  float64 
 9   air_quality_Ozone_num             52542 non-null  float64 
 10  air_quality_Nitrogen_dioxide_num  52542 non-null  float64 
 11  air_quality_Sulphur_dioxide_num   52542 non-null  float6

Các cột từ 15 trở đi sai


### Chuyển kdl = kdl mong muốn + NAN


In [75]:
for col in df.columns.tolist()[15:]:
  print(f"{col} -> {set(map(type, df[col]))}")


moon_phase_nom -> {<class 'str'>}
sunrise_ord -> {<class 'str'>}
moonrise_ord -> {<class 'str'>}
temp_bin_target -> {<class 'str'>}


Tất cả các cột đều đúng kiểu dữ liệu


## Kiểm tra nội dung các cột `binary`


In [76]:
for col in binary_cols:
  print(f"{col} -> {df[col].unique().tolist()}")

Không có cột nào hết


## Kiểm tra nội dung các cột `nominal`


In [77]:
for col in nominal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


moon_phase_nom -> ['Waxing Crescent', 'Waning Gibbous', 'Waning Crescent', 'Last Quarter', 'Waxing Gibbous', 'First Quarter', 'Full Moon', 'New Moon']


## Kiểm tra nội dung các cột `ordinal`


In [78]:
for col in ordinal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


sunrise_ord -> ['07:18 AM', '07:26 AM', '06:18 AM', '06:54 AM', '05:48 AM', '05:45 AM', '07:13 AM', '06:31 AM', '05:32 AM', '06:14 AM', '05:17 AM', '06:03 AM', '06:52 AM', '06:40 AM', '06:41 AM', '06:50 AM', '05:16 AM', '07:30 AM', '07:56 AM', '06:22 AM', '05:38 AM', '07:17 AM', '06:42 AM', '06:57 AM', '05:33 AM', '06:20 AM', '05:41 AM', '05:23 AM', '06:07 AM', '05:57 AM', '06:00 AM', '04:56 AM', '08:43 AM', '06:13 AM', '05:19 AM', '06:34 AM', '07:22 AM', '04:44 AM', '06:19 AM', '06:09 AM', '05:22 AM', '05:30 AM', '06:05 AM', '05:59 AM', '05:54 AM', '06:27 AM', '05:42 AM', '06:12 AM', '07:45 AM', '06:48 AM', '05:40 AM', '06:32 AM', '06:53 AM', '07:19 AM', '05:56 AM', '05:49 AM', '06:17 AM', '07:03 AM', '06:37 AM', '06:08 AM', '06:16 AM', '07:37 AM', '06:43 AM', '07:16 AM', '06:38 AM', '06:01 AM', '05:18 AM', '05:36 AM', '06:39 AM', '05:39 AM', '05:43 AM', '07:54 AM', '04:06 AM', '07:05 AM', '07:31 AM', '07:59 AM', '07:42 AM', '07:50 AM', '06:45 AM', '05:21 AM', '06:06 AM', '05:51 AM', 

In [79]:
# Chuỗi nào có PM thì giá trị giờ cộng thêm 12 phút
def process_time(gold_time: str):
    if gold_time.endswith("PM"):
        parts = gold_time.split(":")
        hour = int(parts[0]) + 12
        gold_time = f"{hour}:{parts[1]}"

    res = re.split("(AM|PM)", gold_time)[0].strip()
    res = res.split(":")[0]
    return res

format = r"\d+:\d+\s*(AM|PM)"

df_ordinal_cols = df[ordinal_cols]
index_not_satisfy_format = (
    df_ordinal_cols[
        df_ordinal_cols.applymap(
            lambda item: re.fullmatch(format, item) is None
        )
    ]
    .stack()
    .index
)

df_ordinal_cols_happen_stack = df_ordinal_cols.stack()
df_ordinal_cols_happen_stack = df_ordinal_cols_happen_stack[
    ~df_ordinal_cols_happen_stack.index.isin(index_not_satisfy_format)
]
df_ordinal_cols_happen_stack = df_ordinal_cols_happen_stack.apply(
    lambda item: process_time(item)
)
df_ordinal_cols_stack = df_ordinal_cols.stack()
df_ordinal_cols_stack[df_ordinal_cols_happen_stack.index] = (
    df_ordinal_cols_happen_stack
)
df[ordinal_cols] = df_ordinal_cols_stack.unstack()

for col in ordinal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")

<ipython-input-79-943485b61a46>:17: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



sunrise_ord -> ['07', '06', '05', '04', '08', '03', '09', '10', '11', '02']
moonrise_ord -> ['07', '14', '23', '05', '01', '18', '20', '03', '08', '24', 'No moonrise', '22', '21', '16', '11', '10', '09', '12', '04', '02', '06', '19', '13', '17', '15']


In [80]:
col_name = 'sunrise_ord'
replace_value = [
    [['02', '03', '04', '05'], 'early_morning'],
    [['06','07', '08', '09' ], 'morning'],
    [['10', '11'], 'midday']
]
df[col_name] = myfuncs.replace_in_category_series_33(df[col_name], replace_value)
df[col_name].unique()

['morning', 'early_morning', 'midday']
Categories (3, string): [early_morning, midday, morning]

In [81]:
col_name = 'moonrise_ord'
replace_value = [
    [['02','03', '04', '05'], 'early_morning'],
    [['06','07', '08', '09'], 'morning'],
    [['10','11', '12', '13'], 'midday'],
    [['14','15', '16', '17'], 'afternoon'],
    [['18','19', '20', '21', '22'], 'evening'],
    [['23','24', '01'], 'midnight'],
]
df[col_name] = myfuncs.replace_in_category_series_33(df[col_name], replace_value)
df[col_name].unique()

['morning', 'afternoon', 'midnight', 'early_morning', 'evening', 'No moonrise', 'midday']
Categories (7, string): [No moonrise, afternoon, early_morning, evening, midday, midnight, morning]

## Kiểm tra nội dung các cột `target`


In [82]:
print(f"{target_col} -> {df[target_col].unique().tolist()}")


temp_bin_target -> ['low', 'medium', 'high']


## Fill missing value


In [83]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="mean"), numeric_cols),
        ("numCat", SimpleImputer(strategy="most_frequent"), numericCat_cols),
        ("cat", SimpleImputer(strategy="most_frequent"), cat_cols),
        ("target", SimpleImputer(strategy="most_frequent"), [target_col]),
    ]
)

df = preprocessor.fit_transform(df)

df = pd.DataFrame(
    df, columns=numeric_cols + numericCat_cols + cat_cols + [target_col]
)

df.head()


,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,air_quality_Nitrogen_dioxide_num,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,sunrise_ord,moonrise_ord,temp_bin_target
0,18.0,57.0,30.39,81.0,75.0,10.0,1.3,24.2,384.8,52.0,17.205,0.925,21.09,23.495,1.0,Waxing Crescent,morning,morning,low
1,10.1,247.0,30.55,80.0,2.0,10.0,0.3,17.0,362.6,26.0,31.82,0.925,17.575,29.23,37.0,Waxing Crescent,morning,afternoon,low
2,24.5,75.0,29.83,89.0,100.0,19.0,0.0,33.6,177.6,31.0,0.925,0.925,5.92,9.065,69.0,Waning Gibbous,morning,midnight,medium
3,22.3,211.0,30.02,12.0,0.0,10.0,6.0,25.7,240.3,76.5,1.1,3.5,5.7,13.8,5.0,Waning Crescent,morning,early_morning,low
4,3.6,256.0,29.94,40.0,0.0,10.0,6.0,3.7,264.55,80.0,8.695,6.29,8.14,13.32,71.0,Waning Gibbous,early_morning,midnight,medium


## Chuyển đổi các cột về đúng kiểu dữ liệu


In [84]:
df[numeric_cols] = df[numeric_cols].astype("float32")
df[numericCat_cols] = df[numericCat_cols].astype("float32")
df[cat_cols] = df[cat_cols].astype("category")
df[target_col] = df[target_col].astype("category")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52542 entries, 0 to 52541
Data columns (total 19 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   wind_kph_num                      52542 non-null  float32 
 1   wind_degree_num                   52542 non-null  float32 
 2   pressure_in_num                   52542 non-null  float32 
 3   humidity_num                      52542 non-null  float32 
 4   cloud_num                         52542 non-null  float32 
 5   visibility_km_num                 52542 non-null  float32 
 6   uv_index_num                      52542 non-null  float32 
 7   gust_kph_num                      52542 non-null  float32 
 8   air_quality_Carbon_Monoxide_num   52542 non-null  float32 
 9   air_quality_Ozone_num             52542 non-null  float32 
 10  air_quality_Nitrogen_dioxide_num  52542 non-null  float32 
 11  air_quality_Sulphur_dioxide_num   52542 non-null  floa

## Loại bỏ duplicates


In [85]:
print(f"Tỉ lệ duplicates: {len(df.index[df.duplicated()]) / len(df.index) * 100}" )
df = df.drop_duplicates().reset_index()

Tỉ lệ duplicates: 0.41109969167523125


In [86]:
df.head()

,index,wind_kph_num,wind_degree_num,pressure_in_num,humidity_num,cloud_num,visibility_km_num,uv_index_num,gust_kph_num,air_quality_Carbon_Monoxide_num,air_quality_Ozone_num,air_quality_Nitrogen_dioxide_num,air_quality_Sulphur_dioxide_num,air_quality_PM2_5_num,air_quality_PM10_num,moon_illumination_num,moon_phase_nom,sunrise_ord,moonrise_ord,temp_bin_target
0,0,18.000000,57.0,30.389999,81.0,75.0,10.0,1.3,24.200001,384.799988,52.0,17.205,0.925,21.090000,23.495001,1.0,Waxing Crescent,morning,morning,low
1,1,10.100000,247.0,30.549999,80.0,2.0,10.0,0.3,17.000000,362.600006,26.0,31.820,0.925,17.575001,29.230000,37.0,Waxing Crescent,morning,afternoon,low
2,2,24.500000,75.0,29.830000,89.0,100.0,19.0,0.0,33.599998,177.600006,31.0,0.925,0.925,5.920000,9.065000,69.0,Waning Gibbous,morning,midnight,medium
3,3,22.299999,211.0,30.020000,12.0,0.0,10.0,6.0,25.700001,240.300003,76.5,1.100,3.500,5.700000,13.800000,5.0,Waning Crescent,morning,early_morning,low
4,4,3.600000,256.0,29.940001,40.0,0.0,10.0,6.0,3.700000,264.549988,80.0,8.695,6.290,8.140000,13.320000,71.0,Waning Gibbous,early_morning,midnight,medium


In [87]:
df = df.drop(columns = ['index'])

## FEATURE_ORDINAL_DICT


In [88]:
binary_cols + ordinal_cols

['sunrise_ord', 'moonrise_ord']

In [89]:
df['moonrise_ord'].unique()

['morning', 'afternoon', 'midnight', 'early_morning', 'evening', 'No moonrise', 'midday']
Categories (7, object): ['No moonrise', 'afternoon', 'early_morning', 'evening', 'midday', 'midnight',
                         'morning']

In [90]:
FEATURE_ORDINAL_DICT = {
  "sunrise_ord": ['early_morning', 'morning', 'midday'],
  "moonrise_ord": ['No moonrise', 'early_morning', 'morning', 'midday', 'afternoon', 'evening', 'midnight'],
}

Fill các giá trị < 0 của 2 cột air_quality_Carbon_Monoxide_num và air_quality_Sulphur_dioxide_num bằng median

In [91]:
col_names = ["air_quality_Carbon_Monoxide_num", "air_quality_Sulphur_dioxide_num"]

for col in col_names:
  df.loc[df.index[df[col] < 0], col] = np.percentile(df[col], 50)

df[numeric_cols].describe().loc[['min', 'max']].T

,min,max
wind_kph_num,3.600000,2963.199951
wind_degree_num,1.000000,360.000000
pressure_in_num,27.959999,88.589996
humidity_num,2.000000,100.000000
cloud_num,0.000000,100.000000
visibility_km_num,0.000000,32.000000
uv_index_num,0.000000,16.299999
gust_kph_num,3.600000,2970.399902
air_quality_Carbon_Monoxide_num,81.000000,38879.398438
air_quality_Ozone_num,0.000000,480.700012


Fill các giá trị outliers của các cột wind_kph_num, pressure_in_num, visibility_km_num, gust_kph_num, air_quality_Carbon_Monoxide_num, air_quality_Ozone_num, air_quality_Nitrogen_dioxide_num, air_quality_Sulphur_dioxide_num, air_quality_PM2_5_num, air_quality_PM10_num bằng median của cột

In [92]:
col_names = [
    'wind_kph_num',
    'pressure_in_num',
    'visibility_km_num',
    'gust_kph_num',
    'air_quality_Carbon_Monoxide_num',
    'air_quality_Ozone_num',
    'air_quality_Nitrogen_dioxide_num',
    'air_quality_Sulphur_dioxide_num',
    'air_quality_PM2_5_num',
    'air_quality_PM10_num',
]

for col_name in col_names:
  df[col_name] = myfuncs.replace_outliers_with_new_value_34(df[col_name], np.percentile(df[col_name], 50))

/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1607: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [94]:
df_copy = myfuncs.load_python_object("artifacts/data_correction_dc4/data.pkl")

fig = px.box(
    y=df_copy['wind_kph_num'],
)

fig.show()

In [93]:
myfuncs.save_python_object("artifacts/data_correction_dc4/data.pkl", df)

# r